In [1]:
import os
import numpy as np
import pandas as pd
import statsmodels.formula.api as smf


def pairwise_mixed_lm(results_dict, cond_a, cond_b, score_col):
    df_a = results_dict[cond_a][["baseline_id", score_col]].assign(condition=cond_a)
    df_b = results_dict[cond_b][["baseline_id", score_col]].assign(condition=cond_b)
    long_df = pd.concat([df_a, df_b], axis=0).rename(columns={score_col: "score"})
    model = smf.mixedlm("score ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
    levels = sorted([cond_a, cond_b])
    term = f"condition[T.{levels[1]}]"
    return {
        "coef":   model.params.get(term, np.nan),
        "stat":   model.tvalues.get(term, np.nan),
        "pvalue": model.pvalues.get(term, np.nan),
        "ref":    levels[0],
        "test":   levels[1],
    }


def compute_all_sig(results_dict, score_col):
    sig = {}
    distractor_conditions = [c for c in results_dict if c != "baseline" and results_dict[c] is not None]

    # vs baseline
    for cond in distractor_conditions:
        df = results_dict[cond]
        long_df = pd.concat([
            df[["baseline_id", score_col]].rename(columns={score_col: "score"}).assign(condition="distractor"),
            df[["baseline_id", f"{score_col}_baseline"]].rename(columns={f"{score_col}_baseline": "score"}).assign(condition="baseline"),
        ], axis=0)
        model = smf.mixedlm("score ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        term = "condition[T.distractor]"
        sig[cond] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
        }

    # pairwise between distractor conditions
    for i, ca in enumerate(distractor_conditions):
        for cb in distractor_conditions[i+1:]:
            key = f"{ca}_vs_{cb}"
            sig[key] = pairwise_mixed_lm(
                {ca: results_dict[ca].rename(columns={score_col: score_col}),
                 cb: results_dict[cb].rename(columns={score_col: score_col})},
                ca, cb, score_col
            )
    return sig


def fmt_pct(v):
    return f"{v*100:.2f}\\%"

def color_min_max(val_dict):
    """Given {col: float}, return {col: latex_string} with red/green for min/max."""
    mn = min(val_dict.values())
    mx = max(val_dict.values())
    out = {}
    for k, v in val_dict.items():
        s = fmt_pct(v)
        if v == mx:
            s = f"\\textcolor{{mygreen}}{{{s}}}"
        elif v == mn:
            s = f"\\textcolor{{red}}{{{s}}}"
        out[k] = s
    return out


def build_latex_table(rows, col_headers, caption, label):
    """
    rows: list of lists of strings (each inner list = one table row)
    col_headers: list of column header strings
    """
    n_cols = len(col_headers)
    col_fmt = "l" * (n_cols - 4) + "cccc"  # last 4 are numeric
    header = " & ".join(f"\\textbf{{{h}}}" for h in col_headers) + " \\\\"
    body = "\n".join(" & ".join(r) + " \\\\" for r in rows)
    return "\n".join([
        "\\begin{table*}[!h]",
        "\\centering",
        "\\small",
        f"\\begin{{tabular}}{{{col_fmt}}}",
        "\\toprule",
        header,
        "\\midrule",
        body,
        "\\bottomrule",
        "\\end{tabular}",
        f"\\caption{{{caption}}}",
        f"\\label{{{label}}}",
        "\\end{table*}",
    ])

In [ ]:
def calculate_results_moralchoice(response_df_filename, scenario_df_filename,
                                   distractor_df_filename="data/distractors.csv"):
    response_df = pd.read_csv(root + response_df_filename)

    response_df["baseline_id"] = (response_df["scenario_id"].astype(str)
                                  + response_df["question_ordering"].astype(str))
    invalid_ids = response_df.loc[
        (response_df["a1_prob"] + response_df["a2_prob"] == 0), "baseline_id"
    ].tolist()
    response_df = response_df.loc[~response_df["baseline_id"].isin(invalid_ids)]
    response_df = response_df.loc[(response_df["a1_prob"] + response_df["a2_prob"] != 0)]

    scenario_df  = pd.read_csv(root + scenario_df_filename)
    distractor_df = pd.read_csv(root + distractor_df_filename)
    response_df = response_df.merge(scenario_df,   left_on="scenario_id", right_on="id", how="left")
    response_df = response_df.merge(distractor_df, left_on="distractor_id", right_on="id", how="left")

    distractor_dfs = {
        "baseline": response_df.loc[pd.isna(response_df["sentiment"])],
        "positive": response_df.loc[response_df["sentiment"] == "positive"],
        "neutral":  response_df.loc[response_df["sentiment"] == "neutral"],
        "negative": response_df.loc[response_df["sentiment"] == "negative"],
    }

    rules = ["death", "pain", "disable", "freedom", "pleasure",
             "deceive", "cheat", "break_promise", "break_law", "duty"]
    results = {k: None for k in distractor_dfs}

    for k, df in distractor_dfs.items():
        for rule in rules:
            a1_col, a2_col = f"a1_{rule}", f"a2_{rule}"
            df_fc = df.loc[
                (((df[a1_col] == "No") & (df[a2_col] == "Yes")) |
                 ((df[a1_col] == "Yes") & (df[a2_col] == "No")))
            ].copy()
            df_fc = df_fc.merge(
                distractor_dfs["baseline"][["baseline_id", "a1_prob", "a2_prob"]],
                on="baseline_id", how="left", suffixes=["_distractor", "_baseline"]
            )
            if len(df_fc) == 0:
                continue

            for suffix in ["distractor", "baseline"]:
                df_fc[f"prob_good_{suffix}"] = np.where(
                    df_fc[a1_col] == "No",
                    df_fc[f"a1_prob_{suffix}"], df_fc[f"a2_prob_{suffix}"]
                )
                df_fc[f"prob_bad_{suffix}"] = np.where(
                    df_fc[a1_col] == "No",
                    df_fc[f"a2_prob_{suffix}"], df_fc[f"a1_prob_{suffix}"]
                )
                df_fc[f"mmap_{suffix}"] = (
                    df_fc[f"prob_good_{suffix}"]
                    / (df_fc[f"prob_good_{suffix}"] + df_fc[f"prob_bad_{suffix}"])
                )

            df_fc["mmap_diff"] = df_fc["mmap_distractor"] - df_fc["mmap_baseline"]
            df_fc = df_fc[["scenario_id", "baseline_id", "mmap_distractor", "mmap_baseline", "mmap_diff"]]

            results[k] = df_fc if results[k] is None else pd.concat([results[k], df_fc])

    mean_mmaps, st_error_mmaps, mean_mmap_diffs, st_error_mmap_diffs = {}, {}, {}, {}
    for cond, df in results.items():
        mean_mmaps[cond]        = np.mean(df["mmap_distractor"])
        st_error_mmaps[cond]    = np.std(df["mmap_distractor"]) / np.sqrt(len(df))
        mean_mmap_diffs[cond]   = np.mean(df["mmap_diff"])
        st_error_mmap_diffs[cond] = np.std(df["mmap_diff"]) / np.sqrt(len(df))

    sig_mmaps = {}
    for cond in ["positive", "neutral", "negative"]:
        df = results[cond]
        long_df = pd.concat([
            df[["baseline_id", "mmap_distractor"]].rename(columns={"mmap_distractor": "mmap"}).assign(condition="distractor"),
            df[["baseline_id", "mmap_baseline"]].rename(columns={"mmap_baseline": "mmap"}).assign(condition="baseline"),
        ])
        model = smf.mixedlm("mmap ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        term = "condition[T.distractor]"
        sig_mmaps[cond] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
        }

    # pairwise between distractor conditions
    for ca, cb in [("negative", "neutral"), ("positive", "neutral"), ("positive", "negative")]:
        df_a = results[ca][["baseline_id", "mmap_distractor"]].assign(condition=ca)
        df_b = results[cb][["baseline_id", "mmap_distractor"]].assign(condition=cb)
        long_df = pd.concat([df_a, df_b]).rename(columns={"mmap_distractor": "mmap"})
        model = smf.mixedlm("mmap ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        levels = sorted([ca, cb])
        term = f"condition[T.{levels[1]}]"
        sig_mmaps[f"{ca}_vs_{cb}"] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "ref": levels[0], "test": levels[1],
        }

    return mean_mmaps, mean_mmap_diffs, st_error_mmaps, st_error_mmap_diffs, sig_mmaps, results

In [2]:
def calculate_results_normbank(response_df_filename,
                                distractor_df_filename="data/distractors.csv"):
    """
    Score = good_prob - bad_prob  (higher = more moral).
    No rule-level forced-choice filtering; baseline_id = scenario_id (no question ordering).
    """
    response_df   = pd.read_csv(root + response_df_filename)
    distractor_df = pd.read_csv(root + distractor_df_filename)
    response_df   = response_df.merge(distractor_df, left_on="distractor_id", right_on="id", how="left")

    response_df["baseline_id"] = response_df["scenario_id"].astype(str)
    response_df["score"] = response_df["good_prob"] - response_df["bad_prob"]

    # drop rows where all probs are zero
    invalid = response_df.loc[
        (response_df["good_prob"] + response_df["ok_prob"] + response_df["bad_prob"] == 0),
        "baseline_id"
    ].tolist()
    response_df = response_df.loc[~response_df["baseline_id"].isin(invalid)]

    distractor_dfs = {
        "baseline": response_df.loc[pd.isna(response_df["sentiment"])].copy(),
        "positive": response_df.loc[response_df["sentiment"] == "positive"].copy(),
        "neutral":  response_df.loc[response_df["sentiment"] == "neutral"].copy(),
        "negative": response_df.loc[response_df["sentiment"] == "negative"].copy(),
    }

    results = {}
    for k, df in distractor_dfs.items():
        base = distractor_dfs["baseline"][["baseline_id", "score"]].rename(columns={"score": "score_baseline"})
        merged = df.merge(base, on="baseline_id", how="left")
        merged = merged.rename(columns={"score": "score_distractor"})
        merged["score_diff"] = merged["score_distractor"] - merged["score_baseline"]
        results[k] = merged[["scenario_id", "baseline_id", "score_distractor", "score_baseline", "score_diff"]]

    mean_scores, st_error_scores, mean_diffs, st_error_diffs = {}, {}, {}, {}
    for cond, df in results.items():
        mean_scores[cond]    = np.mean(df["score_distractor"])
        st_error_scores[cond] = np.std(df["score_distractor"]) / np.sqrt(len(df))
        mean_diffs[cond]     = np.mean(df["score_diff"])
        st_error_diffs[cond] = np.std(df["score_diff"]) / np.sqrt(len(df))

    sig = {}
    for cond in ["positive", "neutral", "negative"]:
        df = results[cond]
        long_df = pd.concat([
            df[["baseline_id", "score_distractor"]].rename(columns={"score_distractor": "score"}).assign(condition="distractor"),
            df[["baseline_id", "score_baseline"]].rename(columns={"score_baseline": "score"}).assign(condition="baseline"),
        ])
        model = smf.mixedlm("score ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        term = "condition[T.distractor]"
        sig[cond] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
        }

    for ca, cb in [("negative", "neutral"), ("positive", "neutral"), ("positive", "negative")]:
        df_a = results[ca][["baseline_id", "score_distractor"]].assign(condition=ca)
        df_b = results[cb][["baseline_id", "score_distractor"]].assign(condition=cb)
        long_df = pd.concat([df_a, df_b]).rename(columns={"score_distractor": "score"})
        model = smf.mixedlm("score ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        levels = sorted([ca, cb])
        term = f"condition[T.{levels[1]}]"
        sig[f"{ca}_vs_{cb}"] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "ref": levels[0], "test": levels[1],
        }

    return mean_scores, mean_diffs, st_error_scores, st_error_diffs, sig, results

In [3]:
def calculate_results_reddit(response_df_filename,
                              distractor_df_filename="data/distractors.csv"):
    response_df   = pd.read_csv(root + response_df_filename)
    distractor_df = pd.read_csv(root + distractor_df_filename)
    response_df   = response_df.merge(distractor_df, left_on="distractor_id", right_on="id", how="left")

    response_df["baseline_id"] = response_df["scenario_id"].astype(str)

    prob_cols = ["esh_prob", "yta_prob", "nta_prob", "nah_prob", "info_prob"]
    total = response_df[prob_cols].sum(axis=1)
    invalid = response_df.loc[total == 0, "baseline_id"].tolist()
    response_df = response_df.loc[~response_df["baseline_id"].isin(invalid)]

    response_df["score"] = (
        response_df["nta_prob"] + response_df["nah_prob"]
        - response_df["yta_prob"] - response_df["esh_prob"]
    )

    distractor_dfs = {
        "baseline": response_df.loc[pd.isna(response_df["sentiment"])].copy(),
        "positive": response_df.loc[response_df["sentiment"] == "positive"].copy(),
        "neutral":  response_df.loc[response_df["sentiment"] == "neutral"].copy(),
        "negative": response_df.loc[response_df["sentiment"] == "negative"].copy(),
    }

    results = {}
    for k, df in distractor_dfs.items():
        base = distractor_dfs["baseline"][["baseline_id", "score"]].rename(columns={"score": "score_baseline"})
        merged = df.merge(base, on="baseline_id", how="left")
        merged = merged.rename(columns={"score": "score_distractor"})
        merged["score_diff"] = merged["score_distractor"] - merged["score_baseline"]
        results[k] = merged[["scenario_id", "baseline_id", "score_distractor", "score_baseline", "score_diff"]]

    mean_scores, st_error_scores, mean_diffs, st_error_diffs = {}, {}, {}, {}
    for cond, df in results.items():
        mean_scores[cond]     = np.mean(df["score_distractor"])
        st_error_scores[cond] = np.std(df["score_distractor"]) / np.sqrt(len(df))
        mean_diffs[cond]      = np.mean(df["score_diff"])
        st_error_diffs[cond]  = np.std(df["score_diff"]) / np.sqrt(len(df))

    sig = {}
    for cond in ["positive", "neutral", "negative"]:
        df = results[cond]
        long_df = pd.concat([
            df[["baseline_id", "score_distractor"]].rename(columns={"score_distractor": "score"}).assign(condition="distractor"),
            df[["baseline_id", "score_baseline"]].rename(columns={"score_baseline": "score"}).assign(condition="baseline"),
        ])
        model = smf.mixedlm("score ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        term = "condition[T.distractor]"
        sig[cond] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
        }

    for ca, cb in [("negative", "neutral"), ("positive", "neutral"), ("positive", "negative")]:
        df_a = results[ca][["baseline_id", "score_distractor"]].assign(condition=ca)
        df_b = results[cb][["baseline_id", "score_distractor"]].assign(condition=cb)
        long_df = pd.concat([df_a, df_b]).rename(columns={"score_distractor": "score"})
        model = smf.mixedlm("score ~ condition", long_df, groups=long_df["baseline_id"]).fit(disp=False)
        levels = sorted([ca, cb])
        term = f"condition[T.{levels[1]}]"
        sig[f"{ca}_vs_{cb}"] = {
            "coef":   model.params.get(term, np.nan),
            "stat":   model.tvalues.get(term, np.nan),
            "pvalue": model.pvalues.get(term, np.nan),
            "ref": levels[0], "test": levels[1],
        }

    return mean_scores, mean_diffs, st_error_scores, st_error_diffs, sig, results

In [ ]:
MODELS = ["gemma-3-4b-it", "Llama-3.2-3B-Instruct", "Qwen3-4B", "GPT-4.1"]
MODEL_LABEL = {
    "gemma-3-4b-it":         "\\lm{gemma-3-4b-it}",
    "Llama-3.2-3B-Instruct": "\\lm{Llama-3.2-3B-Instruct}",
    "Qwen3-4B":              "\\lm{Qwen3-4B}",
    "GPT-4.1":               "\\lm{GPT-4.1}",
}

# (dataset_key, figure_name, call_fn, positional_args)
dataset_configs = {
    "moralchoice_high_ambiguity": [
        ("gemma-3-4b-it",         calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_high_ambiguity/google_gemma-3-4b-it_moralchoice_high_ambiguity.csv",
          "data/scenarios/moralchoice_high_ambiguity.csv")),
        ("Llama-3.2-3B-Instruct", calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_high_ambiguity/meta-llama_Llama-3.2-3B-Instruct_moralchoice_high_ambiguity.csv",
          "data/scenarios/moralchoice_high_ambiguity.csv")),
        ("Qwen3-4B",              calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_high_ambiguity/Qwen_Qwen3-4B_moralchoice_high_ambiguity.csv",
          "data/scenarios/moralchoice_high_ambiguity.csv")),
        ("GPT-4.1",               calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_high_ambiguity/openai_gpt-4.1_moralchoice_high_ambiguity.csv",
          "data/scenarios/moralchoice_high_ambiguity.csv")),
    ],
    "moralchoice_low_ambiguity": [
        ("gemma-3-4b-it",         calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_low_ambiguity/google_gemma-3-4b-it_moralchoice_low_ambiguity.csv",
          "data/scenarios/moralchoice_low_ambiguity.csv")),
        ("Llama-3.2-3B-Instruct", calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_low_ambiguity/meta-llama_Llama-3.2-3B-Instruct_moralchoice_low_ambiguity.csv",
          "data/scenarios/moralchoice_low_ambiguity.csv")),
        ("Qwen3-4B",              calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_low_ambiguity/Qwen_Qwen3-4B_moralchoice_low_ambiguity.csv",
          "data/scenarios/moralchoice_low_ambiguity.csv")),
        ("GPT-4.1",               calculate_results_moralchoice,
         ("data/csv_results/main/moralchoice_low_ambiguity/openai_gpt-4.1_moralchoice_low_ambiguity.csv",
          "data/scenarios/moralchoice_low_ambiguity.csv")),
    ],
    "normbank": [
        ("gemma-3-4b-it",         calculate_results_normbank,
         ("data/csv_results/main/normbank/google_gemma-3-4b-it_normbank.csv",)),
        ("Llama-3.2-3B-Instruct", calculate_results_normbank,
         ("data/csv_results/main/normbank/meta-llama_Llama-3.2-3B-Instruct_normbank.csv",)),
        ("Qwen3-4B",              calculate_results_normbank,
         ("data/csv_results/main/normbank/Qwen_Qwen3-4B_normbank.csv",)),
        ("GPT-4.1",               calculate_results_normbank,
         ("data/csv_results/main/normbank/openai_gpt-4.1_normbank.csv",)),
    ],
    "reddit": [
        ("gemma-3-4b-it",         calculate_results_reddit,
         ("data/csv_results/main/reddit/google_gemma-3-4b-it_reddit.csv",)),
        ("Llama-3.2-3B-Instruct", calculate_results_reddit,
         ("data/csv_results/main/reddit/meta-llama_Llama-3.2-3B-Instruct_reddit.csv",)),
        ("Qwen3-4B",              calculate_results_reddit,
         ("data/csv_results/main/reddit/Qwen_Qwen3-4B_reddit.csv",)),
        ("GPT-4.1",               calculate_results_reddit,
         ("data/csv_results/main/reddit/openai_gpt-4.1_reddit.csv",)),
    ],
}

all_results = {}

for dataset, entries in dataset_configs.items():
    all_results[dataset] = {}
    for model_key, fn, args in entries:
        print(f"  Running {dataset} / {model_key} ...", end=" ")
        mean_scores, mean_diffs, st_errors, st_error_diffs, sig, raw = fn(*args)
        all_results[dataset][model_key] = {
            "mean_scores":     mean_scores,
            "mean_diffs":      mean_diffs,
            "st_errors":       st_errors,
            "st_error_diffs":  st_error_diffs,
            "sig":             sig,
        }
        print("done")
        print(f"    scores: {mean_scores}")
        print(f"    sig:    {sig}\n")

DATASET_META = {
    "moralchoice_high_ambiguity": {
        "score_label": "MMAP",
        "caption_detail": "high-ambiguity MoralChoice scenarios",
        "label": "tab:main-high-ambiguity",
    },
    "moralchoice_low_ambiguity": {
        "score_label": "MMAP",
        "caption_detail": "low-ambiguity MoralChoice scenarios",
        "label": "tab:main-low-ambiguity",
    },
    "normbank": {
        "score_label": "Score (good$-$bad)",
        "caption_detail": "NormBank scenarios",
        "label": "tab:main-normbank",
    },
    "reddit": {
        "score_label": "Score (NTA+NAH$-$YTA$-$ESH)",
        "caption_detail": "Reddit AITA scenarios",
        "label": "tab:main-reddit",
    },
}

tables = []
col_headers = ["Model", "Baseline", "Positive", "Neutral", "Negative"]

for dataset, model_results in all_results.items():
    meta = DATASET_META[dataset]
    rows = []
    for model_key in MODELS:
        if model_key not in model_results:
            continue
        s = model_results[model_key]["mean_scores"]
        b   = s.get("baseline", float("nan"))
        pos = s.get("positive", float("nan"))
        neu = s.get("neutral",  float("nan"))
        neg = s.get("negative", float("nan"))

        colored = color_min_max({"positive": pos, "neutral": neu, "negative": neg})
        rows.append([
            MODEL_LABEL.get(model_key, model_key),
            fmt_pct(b),
            colored["positive"],
            colored["neutral"],
            colored["negative"],
        ])

    caption = (
        f"Mean {meta['score_label']} across distractor sentiment conditions "
        f"on \\textbf{{{meta['caption_detail']}}}. "
        f"Highest values in \\textcolor{{mygreen}}{{green}}, "
        f"lowest in \\textcolor{{red}}{{red}}."
    )
    tables.append(build_latex_table(rows, col_headers, caption, meta["label"]))

output_path = root + "data/tables/main_results_tables.tex"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
with open(output_path, "w") as f:
    f.write("\n\n".join(tables))

print(f"\nWrote {len(tables)} tables to {output_path}")